In [3]:
import time
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.decomposition import TruncatedSVD
import dask.array as da
from dask.distributed import Client
from functools import partial
import graphlib

# 地区格点形成的矩阵的大小,定义为全局变量
global_var = (720, 360)


# 插值函数
def fill_vector(vector, layer):
    # vector = np.squeeze(vector)
    x = np.arange(len(vector))
    known_indices = (vector >= 0) & (~np.isnan(vector))
    known_x = x[known_indices]
    known_values = vector[known_indices]
    # print(layer)
    interp_func = interp1d(known_x, known_values, kind='linear', bounds_error=False, fill_value=np.nan)
    return interp_func(x)


# 数据处理的条件
def condition(element):
    return 0 <= element < 250


# 对数归一化
def log_normalize(data):
    log_data = np.log1p(data)  # 对数据进行对数变换，使用log1p避免对0取对数的情况
    log_mean = np.mean(log_data)
    normalized_data = log_data / log_mean
    return normalized_data


# 对矩阵元素进行平方归一化
def square_normalize(matrix):
    squared_matrix = np.square(matrix)  # 对矩阵元素进行平方
    sum_of_elements = np.nansum(squared_matrix)  # 计算平方后的矩阵元素之和
    normalized_matrix = matrix / np.sqrt(sum_of_elements)  # 对矩阵元素进行归一化

    return normalized_matrix


# 最大最小值归一化
def max_min_normalize(data):
    min_val, max_val = np.min(data), np.max(data)  # 计算对数变换后的数据的最小值和最大值
    normalized_data = (data - min_val) / (max_val - min_val)  # 对数据进行归一化

    return normalized_data


# 将本征矢,也就是svd后的行向量进行解码
def decode_matrix(data, indices):
    out_matrix = np.full(global_var, np.nan)
    out_matrix[indices[:, 0], indices[:, 1]] = data
    return out_matrix


def decode_matrix_bk(data, indices):
    condition = np.zeros(global_var, dtype=bool)
    condition[indices[:, 0], indices[:, 1]] = True
    out_matrix = da.where(condition, data, np.nan)
    return out_matrix


# 自定义的svd分解函数
def my_svd(m, k):
    m = np.nan_to_num(m, copy=False)
    c = m.T @ m
    k = m @ m.T
    svd = TruncatedSVD(n_components=k)
    u = svd.fit_transform(c)
    v = svd.fit_transform(k)
    return u, v, svd.singular_values_


def condition_block(blocks):
    return blocks


In [4]:
start_time = time.time()  # 记录开始时间
# your code goes here...
# 你的计算代码
# 一些数据的初始化
area_name_list = ['western_pacific', 'eastern_pacific', 'Atlantic_Ocean', 'indian_ocean']
file_name_list = ['split_wp', 'split_ep', 'split_al', 'split_io']
variable_name_list = ['all_st_' + _ for _ in ['wp', 'ep', 'al', 'io']]
out_num = 20

# start_list为每个地区的起始经度, 以及一面的这几个变量用于为画图标经纬度记列标签
start_list = [120, 150, 60, 30]
start_lat = 20
grid_label_x_wp = [str(start_list[0] + i * 0.25)[:-2] + 'E' for i in range(0, 241, 40)] + [
    str(180 - i * 0.25)[:-2] + 'W' for i in range(40, 121, 40)]
grid_label_x_ep = [str(start_list[1] - i * 0.25)[:-2] + 'W' for i in range(0, 361, 40)]
grid_label_x_al = [str(start_list[2] - i * 0.25)[:-2] + 'W' for i in range(0, 241, 40)] + [str(i * 0.25)[:-2] + 'E' for
                                                                                           i in range(40, 121, 40)]
grid_label_x_io = [str(start_list[3] + i * 0.25)[:-2] + 'E' for i in range(0, 361, 40)]
grid_label_y = [str(start_lat - i * 0.25)[:-2] + 'N' for i in range(0, 81, 40)] + [str(i * 0.25)[:-2] + 'W' for i in
                                                                                   range(40, 81, 40)]
grid_label_list_x = [grid_label_x_wp, grid_label_x_ep, grid_label_x_al, grid_label_x_io]

exo_threshold = 0.01

# 存放奇异值的矩阵
singular_list_vapor = np.zeros((4, out_num))
singular_list_rain = np.zeros((4, out_num))

# 数据的读入,因为数据量比较大,所以每次只读入一个地区的变量,处理完后关闭
# 在开始之前，创建一个Dask的Client

In [5]:

client = Client(n_workers=16)

In [6]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 16
Total threads: 32,Total memory: 63.75 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50050,Workers: 16
Dashboard: http://127.0.0.1:8787/status,Total threads: 32
Started: Just now,Total memory: 63.75 GiB
Comm: tcp://127.0.0.1:50141,Total threads: 2
Dashboard: http://127.0.0.1:50144/status,Memory: 3.98 GiB
Nanny: tcp://127.0.0.1:50053,


In [7]:
f = h5py.File(file_name_list[0] + '.mat', 'r')
raw_variables = da.from_array(f[variable_name_list[0]], chunks=(2, 720, 360, 1))
# f.close()



In [8]:
raw_variables

dask.array<array, shape=(2, 720, 360, 3657), dtype=float64, chunksize=(2, 720, 360, 1), chunktype=numpy.ndarray>

In [9]:
tr2 = da.max(raw_variables).compute()
tr2

110.99800109863281

In [10]:
tr = raw_variables.blocks[0, 0, 0, 5].shape.compute()
tr

array([[[[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        ...,

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]]],


       [[[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-999.]],

        [[-999.],
         [-999.],
         [-999.],
         ...,
         [-999.],
         [-999.],
         [-9

In [11]:

# 去掉格点数据中的陆地的点
indices = da.argwhere(
    da.logical_and((0 < raw_variables[0, :, :, 0]) & (raw_variables[0, :, :, 0] <= 250),
                   (0 <= raw_variables[1, :, :, 0]) & (raw_variables[1, :, :, 0] <= 250)))

In [12]:
indices.compute()

array([[ 46, 306],
       [ 47, 265],
       [ 47, 290],
       ...,
       [607, 244],
       [642, 315],
       [643, 318]], dtype=int64)

In [13]:
def condition_blo(block):
    print(block.shape())

In [14]:
for a in range(0, 4):
    # 数据的读入,因为数据量比较大,所以每次只读入一个地区的变量,处理完后关闭
    # 在开始之前，创建一个Dask的Client
    for v in ['vapor', 'rain']:
        # 当你有大量数据需要加载时，你可以使用dask的from_zarr或from_array函数
        with h5py.File(file_name_list[a] + '_filtered_normalized_' + v + '.h5', 'r') as f:
            indices = np.load('area_indices.npy', allow_pickle=True)[a][0]
            filtered_elements = da.from_array(f['filtered_elements'], chunks=(indices.shape[0], 1))
            k = da.dot(filtered_elements, filtered_elements.T)

        with h5py.File(file_name_list[a] + '_filtered_normalized_'+v+'.h5', 'w') as f:
            dset1 = f.create_dataset('filtered_elements_dot', data=k)



2023-05-20 21:18:54,383 - distributed.protocol.pickle - ERROR - Failed to serialize <ToPickle: HighLevelGraph with 1 layers.
 0. 2319078548480
>.
Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", line 63, in dumps
    result = pickle.dumps(x, **dump_kwargs)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\h5py\_hl\base.py", line 368, in __getnewargs__
    raise TypeError("h5py objects cannot be pickled")
TypeError: h5py objects cannot be pickled

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", line 68, in dumps
    pickler.dump(x)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", line 29, in reducer_override
    return deserialize, serialize(obj)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\h5p

TypeError: ('Could not serialize object of type HighLevelGraph', '<ToPickle: HighLevelGraph with 1 layers.\n<dask.highlevelgraph.HighLevelGraph object at 0x21bf49dbe50>\n 0. 2319078548480\n>')

In [21]:
area_indices = np.empty((4, 1), dtype=object)
for a in range(0, 4):
    with h5py.File(file_name_list[a] + '.mat', 'r') as f:
        print(f.keys())
        raw_variables = f[variable_name_list[a]][()]
        indices = np.argwhere(
            np.logical_and((0 < raw_variables[0, :, :, 0]) & (raw_variables[0, :, :, 0] <= 250),
                           (0 <= raw_variables[1, :, :, 0]) & (raw_variables[1, :, :, 0] <= 250)))
        area_indices[a, 0] = indices
        filtered_elements = raw_variables[:, indices[:, 0], indices[:, 1], :]
        # filtered_elements=da.from_array(filtered_elements,chunks=(1,indices,1))
        eox_indices = da.argwhere(
            da.sum(filtered_elements[0, :, :] <= 0, axis=0) > 0.1 * filtered_elements.shape[1]).compute()
        filtered_elements = np.delete(filtered_elements, eox_indices, axis=2)

        filtered_elements[1, :, :] = np.where((filtered_elements[1, :, :] > 0) & (filtered_elements[1, :, :] <= 250),
                                              filtered_elements[1, :, :], np.nan)
        avg_vapor = np.nanmean(filtered_elements[0, :, :], axis=1).reshape((-1, 1))
        avg_rain = np.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))
        filtered_elements_normalized_vapor = filtered_elements[0, :, :] - avg_vapor
        filtered_elements_normalized_rain = filtered_elements[1, :, :] - avg_rain
        filtered_normalized_vapor = square_normalize(filtered_elements_normalized_vapor)
        filtered_normalized_rain = square_normalize(filtered_elements_normalized_rain)
        # 保存 filtered_elements 到一个新的 HDF5 文件
    with h5py.File(file_name_list[a] + '_filtered_normalized_vapor.h5', 'w') as f:
        dset1 = f.create_dataset('filtered_elements', data=filtered_elements_normalized_vapor)
    with h5py.File(file_name_list[a] + '_filtered_normalized_rain.h5', 'w') as f:
        dset2 = f.create_dataset('filtered_elements', data=filtered_elements_normalized_rain)
np.save('area_indices', area_indices)


<KeysViewHDF5 ['all_st_wp']>


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6044\3420291825.py:19: RuntimeWarning: Mean of empty slice
  avg_rain = np.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))


<KeysViewHDF5 ['all_st_ep']>
<KeysViewHDF5 ['all_st_al']>


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6044\3420291825.py:19: RuntimeWarning: Mean of empty slice
  avg_rain = np.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))


<KeysViewHDF5 ['all_st_io']>


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6044\3420291825.py:19: RuntimeWarning: Mean of empty slice
  avg_rain = np.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))


NameError: name 'h5py' is not defined

In [20]:

area_indices[0, 0] = indices


In [11]:
test_da = da.map_blocks(condition_block, raw_variables, dtype=raw_variables.dtype).compute()

2023-05-19 15:37:42,177 - distributed.protocol.pickle - ERROR - Failed to serialize <ToPickle: HighLevelGraph with 1 layers.
 0. 2289164372160
>.
Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", line 77, in dumps
    result = cloudpickle.dumps(x, **dump_kwargs)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\cloudpickle\cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\cloudpickle\cloudpickle_fast.py", line 632, in dump
    return Pickler.dump(self, obj)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\h5py\_hl\base.py", line 368, in __getnewargs__
    raise TypeError("h5py objects cannot be pickled")
TypeError: h5py objects cannot be pickled

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", li

TypeError: ('Could not serialize object of type HighLevelGraph', '<ToPickle: HighLevelGraph with 1 layers.\n<dask.highlevelgraph.HighLevelGraph object at 0x214fc18a0a0>\n 0. 2289164372160\n>')

In [ ]:
select_elements_with_indices = partial(condition_block, indices=indices)
filtered_elements = da.map_blocks(select_elements_with_indices, raw_variables)

# 找出有大量异常值点的天数去掉
eox_indices = da.unique(
    da.argwhere(da.sum(filtered_elements[0, :, :] <= 0, axis=0) > 0.05 * filtered_elements.shape[1]))
filtered_elements = da.delete(filtered_elements, eox_indices, axis=2)


In [ ]:
# filled_data_vapor = np.array([fill_vector(filtered_elements[0, :, layer], layer) for layer in range(filtered_elements.shape[-1])])
# filled_data_rain = np.array([fill_vector(filtered_elements[1, :, layer], layer) for layer in range(filtered_elements.shape[-1])])

# 删除部分缺失数据比较多,无法进行插值的数据
# filled_data_rain = np.delete(filled_data_rain, np.unique(np.argwhere(np.isnan(filled_data_rain))[:, 0]), axis=0).T
# filled_data_vapor = np.delete(filled_data_vapor, np.unique(np.argwhere(np.isnan(filled_data_vapor))[:, 0]), axis=0).T

# 对数据进行归一化操作,注意这里的降雨求平均有好几种方式
# 这里尝试一下对所有值求平均
filtered_elements[0, :, :] = da.where((filtered_elements[0, :, :] > 0) & (filtered_elements[0, :, :] <= 250),
                                      filtered_elements[0, :, :], np.nan)
filtered_elements[1, :, :] = da.where((filtered_elements[1, :, :] > 0) & (filtered_elements[1, :, :] <= 250),
                                      filtered_elements[1, :, :], np.nan)
# filtered_elements[0, :, :][(filtered_elements[0, :, :] <= 0) | (filtered_elements[0, :, :] > 250)] = np.nan
# filtered_elements[1, :, :][(filtered_elements[1, :, :] < 0) | (filtered_elements[1, :, :] > 250)] = np.nan
avg_vapor = da.nanmean(filtered_elements[0, :, :], axis=1).reshape((-1, 1))
avg_rain = da.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))
filtered_elements_normalized_vapor = filtered_elements[0, :, :] - avg_vapor
filtered_elements_normalized_rain = filtered_elements[1, :, :] - avg_rain
filtered_normalized_vapor = square_normalize(filtered_elements_normalized_vapor)
filtered_normalized_rain = square_normalize(filtered_elements_normalized_rain)

# 进行奇异值分解并将数据保存
reduced_matrix_rain, Vr, singular_list_rain = my_svd(filtered_normalized_rain, out_num)
reduced_matrix_vapor, Vv, singular_list_vapor = my_svd(filtered_normalized_vapor, out_num)
# svd = TruncatedSVD(n_components=out_num)
# reduced_matrix_vapor = svd.fit_transform(filtered_normalized_vapor)
# reduced_matrix_rain = svd.fit_transform(filtered_normalized_rain)
# singular_list_rain[a, :] = svd.singular_values_
# singular_list_vapor[a, :] = svd.singular_values_

# 从分解后的矩阵提取元素同时将图片保存到文件夹
for i in range(out_num):
    for v in ['vapor', 'rain']:
        if v == 'vapor':
            decode_out_variables = np.flipud(decode_matrix(reduced_matrix_vapor[:, i], indices))
        else:
            decode_out_variables = np.flipud(decode_matrix(reduced_matrix_rain[:, i], indices))
        # 保存解码后的数据
        np.save('decode_' + v + str(a), decode_out_variables)
        # 画图
        stats_ax = sns.heatmap(decode_out_variables, cmap="viridis")
        stats_ax.set_xticks(range(0, 361, 40))
        stats_ax.set_xticklabels(grid_label_list_x[a])
        stats_ax.set_yticks(range(0, 161, 40))
        stats_ax.set_yticklabels(grid_label_y)
        stats_ax.xaxis.set_tick_params(rotation=0)
        save_path = 'pcaPicture6th\\' + area_name_list[a] + '\\' + area_name_list[a] + '_' + str(i) + 'th_' + v + '.png'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close()
f.close()
end_time = time.time()  # 记录结束时间
run_time = end_time - start_time  # 计算运行时间
print("程序运行时间为：", run_time, "秒")




In [ ]:
# @njit
# def process_data(filtered_elements):
#     # 找出有大量异常值点的天数去掉
#     eox_indices = np.unique(np.argwhere(np.sum(filtered_elements[0, :, :] < 0, axis=0) > 1000))
#     filtered_elements = np.delete(filtered_elements, eox_indices, axis=2)
#
#     filled_data_vapor = np.empty_like(filtered_elements[0])
#     filled_data_rain = np.empty_like(filtered_elements[1])
#
#     for layer in range(filtered_elements.shape[-1]):
#         filled_data_vapor[:, layer] = fill_vector(filtered_elements[0, :, layer], layer)
#         filled_data_rain[:, layer] = fill_vector(filtered_elements[1, :, layer], layer)
#
#     # 删除部分缺失数据比较多,无法进行插值的数据
#     filled_data_rain = np.delete(filled_data_rain, np.unique(np.argwhere(np.isnan(filled_data_rain))[:, 0]), axis=0).T
#     filled_data_vapor = np.delete(filled_data_vapor, np.unique(np.argwhere(np.isnan(filled_data_vapor))[:, 0]), axis=0).T
#
#     # 对数据进行归一化操作,注意这里的降雨求平均有好几种方式
#     avg_vapor = np.mean(np.ma.masked_array(filled_data_vapor > 0, mask=filled_data_vapor), axis=1).reshape((-1, 1))
#     avg_rain = np.mean(np.ma.masked_array(filled_data_rain > 0, mask=filled_data_rain), axis=1).reshape((-1, 1))
#     filtered_elements_normalized_vapor = filled_data_vapor - avg_vapor
#     filtered_elements_normalized_rain = filled_data_rain - avg_rain
#     filtered_normalized_vapor = square_normalize(filtered_elements_normalized_vapor)
#     filtered_normalized_rain = square_normalize(filtered_elements_normalized_rain)
#
#     return filtered_normalized_vapor, filtered_normalized_rain
#


In [44]:
def test_block(block):
    print(block.shape())

In [45]:
test_da = da.map_blocks(test_block, raw_variables, dtype=raw_variables.dtype).compute()

2023-05-19 11:06:56,398 - distributed.protocol.pickle - ERROR - Failed to serialize <ToPickle: HighLevelGraph with 1 layers.
 0. 2337058151104
>.
Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", line 77, in dumps
    result = cloudpickle.dumps(x, **dump_kwargs)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\cloudpickle\cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\cloudpickle\cloudpickle_fast.py", line 632, in dump
    return Pickler.dump(self, obj)
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\h5py\_hl\base.py", line 368, in __getnewargs__
    raise TypeError("h5py objects cannot be pickled")
TypeError: h5py objects cannot be pickled

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\protocol\pickle.py", li

TypeError: ('Could not serialize object of type HighLevelGraph', '<ToPickle: HighLevelGraph with 1 layers.\n<dask.highlevelgraph.HighLevelGraph object at 0x220290754c0>\n 0. 2337058151104\n>')

In [ ]:
select_elements_with_indices = partial(condition_block, indices=indices)
filtered_elements = da.map_blocks(select_elements_with_indices, raw_variables)

# 找出有大量异常值点的天数去掉
eox_indices = da.unique(
    da.argwhere(da.sum(filtered_elements[0, :, :] <= 0, axis=0) > 0.05 * filtered_elements.shape[1]))
filtered_elements = da.delete(filtered_elements, eox_indices, axis=2)


In [ ]:
# filled_data_vapor = np.array([fill_vector(filtered_elements[0, :, layer], layer) for layer in range(filtered_elements.shape[-1])])
# filled_data_rain = np.array([fill_vector(filtered_elements[1, :, layer], layer) for layer in range(filtered_elements.shape[-1])])

# 删除部分缺失数据比较多,无法进行插值的数据
# filled_data_rain = np.delete(filled_data_rain, np.unique(np.argwhere(np.isnan(filled_data_rain))[:, 0]), axis=0).T
# filled_data_vapor = np.delete(filled_data_vapor, np.unique(np.argwhere(np.isnan(filled_data_vapor))[:, 0]), axis=0).T

# 对数据进行归一化操作,注意这里的降雨求平均有好几种方式
# 这里尝试一下对所有值求平均
filtered_elements[0, :, :] = da.where((filtered_elements[0, :, :] > 0) & (filtered_elements[0, :, :] <= 250),
                                      filtered_elements[0, :, :], np.nan)
filtered_elements[1, :, :] = da.where((filtered_elements[1, :, :] > 0) & (filtered_elements[1, :, :] <= 250),
                                      filtered_elements[1, :, :], np.nan)
# filtered_elements[0, :, :][(filtered_elements[0, :, :] <= 0) | (filtered_elements[0, :, :] > 250)] = np.nan
# filtered_elements[1, :, :][(filtered_elements[1, :, :] < 0) | (filtered_elements[1, :, :] > 250)] = np.nan
avg_vapor = da.nanmean(filtered_elements[0, :, :], axis=1).reshape((-1, 1))
avg_rain = da.nanmean(filtered_elements[1, :, :], axis=1).reshape((-1, 1))
filtered_elements_normalized_vapor = filtered_elements[0, :, :] - avg_vapor
filtered_elements_normalized_rain = filtered_elements[1, :, :] - avg_rain
filtered_normalized_vapor = square_normalize(filtered_elements_normalized_vapor)
filtered_normalized_rain = square_normalize(filtered_elements_normalized_rain)

# 进行奇异值分解并将数据保存
reduced_matrix_rain, Vr, singular_list_rain = my_svd(filtered_normalized_rain, out_num)
reduced_matrix_vapor, Vv, singular_list_vapor = my_svd(filtered_normalized_vapor, out_num)
# svd = TruncatedSVD(n_components=out_num)
# reduced_matrix_vapor = svd.fit_transform(filtered_normalized_vapor)
# reduced_matrix_rain = svd.fit_transform(filtered_normalized_rain)
# singular_list_rain[a, :] = svd.singular_values_
# singular_list_vapor[a, :] = svd.singular_values_

# 从分解后的矩阵提取元素同时将图片保存到文件夹
for i in range(out_num):
    for v in ['vapor', 'rain']:
        if v == 'vapor':
            decode_out_variables = np.flipud(decode_matrix(reduced_matrix_vapor[:, i], indices))
        else:
            decode_out_variables = np.flipud(decode_matrix(reduced_matrix_rain[:, i], indices))
        # 保存解码后的数据
        np.save('decode_' + v + str(a), decode_out_variables)
        # 画图
        stats_ax = sns.heatmap(decode_out_variables, cmap="viridis")
        stats_ax.set_xticks(range(0, 361, 40))
        stats_ax.set_xticklabels(grid_label_list_x[a])
        stats_ax.set_yticks(range(0, 161, 40))
        stats_ax.set_yticklabels(grid_label_y)
        stats_ax.xaxis.set_tick_params(rotation=0)
        save_path = 'pcaPicture6th\\' + area_name_list[a] + '\\' + area_name_list[a] + '_' + str(i) + 'th_' + v + '.png'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close()
f.close()
end_time = time.time()  # 记录结束时间
run_time = end_time - start_time  # 计算运行时间
print("程序运行时间为：", run_time, "秒")




In [3]:
# @njit
# def process_data(filtered_elements):
#     # 找出有大量异常值点的天数去掉
#     eox_indices = np.unique(np.argwhere(np.sum(filtered_elements[0, :, :] < 0, axis=0) > 1000))
#     filtered_elements = np.delete(filtered_elements, eox_indices, axis=2)
#
#     filled_data_vapor = np.empty_like(filtered_elements[0])
#     filled_data_rain = np.empty_like(filtered_elements[1])
#
#     for layer in range(filtered_elements.shape[-1]):
#         filled_data_vapor[:, layer] = fill_vector(filtered_elements[0, :, layer], layer)
#         filled_data_rain[:, layer] = fill_vector(filtered_elements[1, :, layer], layer)
#
#     # 删除部分缺失数据比较多,无法进行插值的数据
#     filled_data_rain = np.delete(filled_data_rain, np.unique(np.argwhere(np.isnan(filled_data_rain))[:, 0]), axis=0).T
#     filled_data_vapor = np.delete(filled_data_vapor, np.unique(np.argwhere(np.isnan(filled_data_vapor))[:, 0]), axis=0).T
#
#     # 对数据进行归一化操作,注意这里的降雨求平均有好几种方式
#     avg_vapor = np.mean(np.ma.masked_array(filled_data_vapor > 0, mask=filled_data_vapor), axis=1).reshape((-1, 1))
#     avg_rain = np.mean(np.ma.masked_array(filled_data_rain > 0, mask=filled_data_rain), axis=1).reshape((-1, 1))
#     filtered_elements_normalized_vapor = filled_data_vapor - avg_vapor
#     filtered_elements_normalized_rain = filled_data_rain - avg_rain
#     filtered_normalized_vapor = square_normalize(filtered_elements_normalized_vapor)
#     filtered_normalized_rain = square_normalize(filtered_elements_normalized_rain)
#
#     return filtered_normalized_vapor, filtered_normalized_rain
#


F:\ZZBO\miniconda\envs\liusch\lib\site-packages\distributed\node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 65368 instead
  warnings.warn(


KeyboardInterrupt: 